# Classification Penguins - MLflow Integration

In [ ]:
import pandas as pd
import boto3
import io
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

import warnings
warnings.filterwarnings('ignore')

# MLflow Configs
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio:9000"

mlflow.set_tracking_uri("http://mlflow-server:5000")
mlflow.set_experiment("Penguins_Classification")
print("MLflow Tracking URI configurada!")


In [ ]:
# 1. Extract (S3 via pandas and boto3)
s3_client = boto3.client(
    's3', endpoint_url='http://minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    region_name='us-east-1'
)

# Simulando extração da Raw/Landing
response = s3_client.get_object(Bucket='landing-zone', Key='penguins.csv')
df = pd.read_csv(io.BytesIO(response['Body'].read()))
df.dropna(inplace=True)
print("Dados extraídos com sucesso! Amostra:")
df.head(3)

In [ ]:
# 2. Preparação dos Dados
# Converter variáveis categóricas
X = pd.get_dummies(df.drop('especies', axis=1))
y = df['especies']

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)
print("Dados particionados!", X_treino.shape)

In [ ]:
# 3. Treinamento e Registro Automático no MLflow
max_depth_param = 5

with mlflow.start_run() as run:
    print(f"Run ID: {run.info.run_id}")
    
    # Treina o modelo
    modelo = DecisionTreeClassifier(max_depth=max_depth_param, random_state=42)
    modelo.fit(X_treino, y_treino)
    
    # Avaliação
    previsoes = modelo.predict(X_teste)
    acuracia = accuracy_score(y_teste, previsoes)
    f1 = f1_score(y_teste, previsoes, average='weighted')
    
    print(f"Acurácia: {acuracia:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    # Registrando tudo no MLflow! -----------------------------
    mlflow.log_param("max_depth", max_depth_param)
    mlflow.log_param("algoritmo", "DecisionTree")
    
    mlflow.log_metric("acuracia", acuracia)
    mlflow.log_metric("f1_score", f1)
    
    # Salvando a assinatura do modelo e jogando pro S3
    from mlflow.models.signature import infer_signature
    signature = infer_signature(X_treino, modelo.predict(X_treino))
    
    mlflow.sklearn.log_model(
        sk_model=modelo,
        artifact_path="modelo_penguins",
        signature=signature,
        registered_model_name="PenguinClassifier"
    )
    
    print("✅ Parâmetros, Métricas e Modelo Pickle enviados ao MLflow e S3 com sucesso! (http://localhost:5000)")